# V3 · GPU · Proyecto integrador · Equipo NN

**Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1**

Antes de ejecutar: *Entorno de ejecución → Cambiar tipo de entorno → GPU* (Colab) o *Settings → Accelerator → GPU* (Kaggle). Ejecuta las celdas en orden (*Entorno de ejecución → Ejecutar todo*).

Al terminar: (1) descarga este notebook **con las salidas** (*Archivo → Descargar → .ipynb*) y reemplaza `v3/notebook_colab.ipynb`; (2) descarga `mediciones_v3.csv` y `entorno_gpu.txt` a `resultados/` del repositorio. Sin esos tres archivos la V3 no tiene evidencia.

Este notebook lo generó `herramientas/generar_notebook_v3.py`; si cambias el código fuente, regenéralo en lugar de editar las celdas a mano.

In [ ]:
# 1. Entorno: qué GPU nos tocó HOY (cambia entre sesiones; por eso se guarda con fecha)
!nvidia-smi
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv > entorno_gpu.txt
!date -u +fecha_utc=%Y-%m-%dT%H:%M:%SZ >> entorno_gpu.txt
!nvcc --version | tail -1 >> entorno_gpu.txt
!grep -m1 'model name' /proc/cpuinfo | sed 's/^/cpu_anfitrion: /' >> entorno_gpu.txt
!nproc | sed 's/^/nucleos_anfitrion=/' >> entorno_gpu.txt
!cat entorno_gpu.txt

In [ ]:
%%writefile problema.h
/*
 * comun/problema.h · Plantilla del proyecto integrador (Cómputo Paralelo · 6BM1 · 27/1)
 *
 * Qué contiene: lo que comparten V1 (secuencial), V2 (OpenMP) y V3 (GPU, parte de
 * anfitrión): generación determinista de datos a partir de una semilla, suma de
 * verificación (checksum) y el cómputo por elemento.
 *
 * QUÉ DEBES CAMBIAR (marcado con "PLANTILLA"):
 *   - El tipo de dato y la generación de entrada si tu problema no es un arreglo de double.
 *   - La función `elemento()`: es un marcador de posición compute-bound. Entregarla tal
 *     cual vale 0 en "Correctitud" de la rúbrica; es solo para que la infraestructura
 *     (Makefile, pruebas, run_all.sh, CSV) funcione desde el primer commit.
 * QUÉ NO DEBES CAMBIAR:
 *   - La firma `checksum()` ni el generador: las pruebas comparan V2 y V3 contra V1
 *     con los mismos datos en cualquier máquina (rand() NO sirve: depende de la libc).
 *
 * Compila como C11 (gcc) y como C++ (nvcc): no uses conversiones implícitas de void*.
 */
#ifndef PROBLEMA_H
#define PROBLEMA_H

#include <math.h>
#include <stddef.h>
#include <stdint.h>

/* Tamaño por defecto: la V1 debe correr en < 5 s en una laptop de 4 núcleos y en >= 1 s
 * para que la medición sea válida (protocolo §2). Ajusta N_DEFAULT a tu problema. */
#define N_DEFAULT    4000000u
#define SEED_DEFAULT 12345u
/* PLANTILLA: iteraciones del marcador de posición; sube/baja para calibrar el tiempo. */
#define ITER_DEFAULT 12

/* Generador congruencial lineal de 32 bits (constantes de Numerical Recipes).
 * Devuelve un double en [0, 1). Determinista en cualquier plataforma. */
static inline uint32_t lcg_next(uint32_t *state) {
    *state = (*state) * 1664525u + 1013904223u;
    return *state;
}

static inline double lcg_uniform(uint32_t *state) {
    return (double)lcg_next(state) / 4294967296.0; /* 2^32 */
}

/* Llena a[0..n) con la misma secuencia para la misma semilla. */
static inline void init_input(double *a, size_t n, uint32_t seed) {
    uint32_t state = seed;
    for (size_t i = 0; i < n; ++i) {
        a[i] = lcg_uniform(&state);
    }
}

/* PLANTILLA: cómputo por elemento, deliberadamente costoso (compute-bound) para que el
 * speedup sea visible. Sustitúyelo por tu problema. Si tu problema no es "por elemento"
 * (stencil, matriz, grafo...), reemplaza también los ciclos de v1/main.c, v2/main.c y
 * v3/kernel.cu conservando la salida clave=valor. */
static inline double elemento(double x, int iter) {
    double y = x;
    for (int k = 0; k < iter; ++k) {
        y = sin(y) + cos(0.5 * y) * 0.25;
    }
    return y;
}

/* Suma de verificación. En V2/V3 el orden de la suma cambia el redondeo: por eso las
 * pruebas comparan con tolerancia relativa (tests/run_tests.sh, TOL). */
static inline double checksum(const double *x, size_t n) {
    double s = 0.0;
    for (size_t i = 0; i < n; ++i) {
        s += x[i];
    }
    return s;
}

#endif /* PROBLEMA_H */


In [ ]:
%%writefile main_v1.c
/*
 * v1/main.c · V1 secuencial · Plantilla del proyecto integrador
 *
 * Uso:     ./v1 [n] [semilla]
 *          n       tamaño del problema (por defecto N_DEFAULT)
 *          semilla entero sin signo para generar los datos (por defecto SEED_DEFAULT)
 * Salida:  una línea por campo, formato clave=valor (lo leen tests/ y run_all.sh):
 *          version=v1  n=...  semilla=...  hilos=1  tiempo_s=...  checksum=...
 * Código:  0 si todo bien; 2 si los argumentos son inválidos; 1 si falla la memoria.
 * Ejemplo: ./v1 1000000 7
 *
 * Qué se cronometra: SOLO la región de interés (el cómputo). Ni la generación de datos ni
 * la impresión (protocolo §2). Reloj: clock_gettime(CLOCK_MONOTONIC); no uses clock().
 *
 * PLANTILLA: sustituye el ciclo marcado por tu problema. Conserva la salida clave=valor.
 */
#define _POSIX_C_SOURCE 200809L
#include <errno.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#include "problema.h"

static double wall_seconds(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

/* Convierte un argumento a entero sin signo; devuelve 0 si no es válido. */
static int parse_uint(const char *s, unsigned long *out) {
    char *end = NULL;
    errno = 0;
    unsigned long v = strtoul(s, &end, 10);
    if (errno != 0 || end == s || *end != '\0') {
        return 0;
    }
    *out = v;
    return 1;
}

int main(int argc, char **argv) {
    unsigned long n = N_DEFAULT;
    unsigned long seed = SEED_DEFAULT;

    if (argc > 1 && !parse_uint(argv[1], &n)) {
        fprintf(stderr, "error: n inválido: %s\n", argv[1]);
        return 2;
    }
    if (argc > 2 && !parse_uint(argv[2], &seed)) {
        fprintf(stderr, "error: semilla inválida: %s\n", argv[2]);
        return 2;
    }
    if (n == 0) {
        fprintf(stderr, "error: n debe ser mayor que 0\n");
        return 2;
    }

    double *in = (double *)malloc(n * sizeof(double));
    double *out = (double *)malloc(n * sizeof(double));
    if (in == NULL || out == NULL) {
        fprintf(stderr, "error: sin memoria para n=%lu\n", n);
        free(in);
        free(out);
        return 1;
    }
    init_input(in, n, (uint32_t)seed);

    /* ---- región de interés: empieza el cronómetro ---- */
    double t0 = wall_seconds();
    /* PLANTILLA: aquí va tu cómputo secuencial. */
    for (size_t i = 0; i < n; ++i) {
        out[i] = elemento(in[i], ITER_DEFAULT);
    }
    double t1 = wall_seconds();
    /* ---- región de interés: termina el cronómetro ---- */

    double cs = checksum(out, n);

    printf("version=v1\n");
    printf("n=%lu\n", n);
    printf("semilla=%lu\n", seed);
    printf("hilos=1\n");
    printf("tiempo_s=%.6f\n", t1 - t0);
    printf("checksum=%.12e\n", cs);

    free(in);
    free(out);
    return 0;
}


In [ ]:
%%writefile kernel.cu
/*
 * v3/kernel.cu · V3 GPU (CUDA C) · Plantilla del proyecto integrador
 *
 * Uso:     ./v3 [n] [semilla] [hilos_por_bloque]
 * Salida:  clave=valor:
 *          version=v3  n=...  semilla=...  gpu=<nombre>  hilos_por_bloque=...
 *          tiempo_kernel_s=...   (SOLO el kernel: sin copias anfitrión<->dispositivo)
 *          tiempo_total_s=...    (H2D + kernel + D2H; protocolo §2: se reportan ambos)
 *          checksum=...
 * Código:  0 si todo bien; 2 argumentos inválidos; 1 error de CUDA o de memoria.
 * Ejemplo (Colab, GPU T4):  nvcc -O2 -arch=sm_75 -Xcompiler -Wall,-Wextra -I. -o v3 kernel.cu && ./v3 4000000 12345
 * Verificación local sin GPU: make check  (solo compila: nvcc -c -arch=sm_75)
 *
 * Correctitud: el checksum debe coincidir con el de ./v1 n semilla (tolerancia relativa).
 * La celda de verificación del notebook lo comprueba en la misma sesión de Colab.
 *
 * PLANTILLA: sustituye el kernel y, si hace falta, el patrón de lanzamiento. Conserva la
 * salida clave=valor y las DOS mediciones (con y sin transferencias).
 */
#include <cuda_runtime.h>
#include <errno.h>
#include <stdio.h>
#include <stdlib.h>

#include "problema.h"

/* Revisa el código de retorno de cada llamada a la API. "El kernel no dio error" no es
 * evidencia de correctitud (Anexo A, idea errónea 27): además se compara con la CPU. */
#define CUDA_CHECK(call)                                                            \
    do {                                                                            \
        cudaError_t err__ = (call);                                                 \
        if (err__ != cudaSuccess) {                                                 \
            fprintf(stderr, "error CUDA en %s:%d: %s\n", __FILE__, __LINE__,       \
                    cudaGetErrorString(err__));                                     \
            return 1;                                                               \
        }                                                                           \
    } while (0)

/* PLANTILLA: mismo cómputo por elemento que comun/problema.h, en versión de dispositivo.
 * Si cambias `elemento()` en la cabecera, cambia también esta función. */
__device__ static double elemento_dev(double x, int iter) {
    double y = x;
    for (int k = 0; k < iter; ++k) {
        y = sin(y) + cos(0.5 * y) * 0.25;
    }
    return y;
}

__global__ void kernel_elemento(const double *in, double *out, size_t n, int iter) {
    size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        out[i] = elemento_dev(in[i], iter);
    }
}

static int parse_uint(const char *s, unsigned long *out) {
    char *end = NULL;
    errno = 0;
    unsigned long v = strtoul(s, &end, 10);
    if (errno != 0 || end == s || *end != '\0') {
        return 0;
    }
    *out = v;
    return 1;
}

int main(int argc, char **argv) {
    unsigned long n = N_DEFAULT;
    unsigned long seed = SEED_DEFAULT;
    unsigned long block = 256;

    if (argc > 1 && !parse_uint(argv[1], &n)) {
        fprintf(stderr, "error: n inválido: %s\n", argv[1]);
        return 2;
    }
    if (argc > 2 && !parse_uint(argv[2], &seed)) {
        fprintf(stderr, "error: semilla inválida: %s\n", argv[2]);
        return 2;
    }
    if (argc > 3 && (!parse_uint(argv[3], &block) || block == 0 || block > 1024 ||
                     block % 32 != 0)) {
        fprintf(stderr, "error: hilos_por_bloque debe ser múltiplo de 32 en [32, 1024]: %s\n",
                argv[3]);
        return 2;
    }
    if (n == 0) {
        fprintf(stderr, "error: n debe ser mayor que 0\n");
        return 2;
    }

    size_t bytes = (size_t)n * sizeof(double);
    double *h_in = (double *)malloc(bytes);
    double *h_out = (double *)malloc(bytes);
    if (h_in == NULL || h_out == NULL) {
        fprintf(stderr, "error: sin memoria en el anfitrión para n=%lu\n", n);
        free(h_in);
        free(h_out);
        return 1;
    }
    init_input(h_in, (size_t)n, (uint32_t)seed);

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));

    double *d_in = NULL;
    double *d_out = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_in, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_out, bytes));

    cudaEvent_t ev_total0, ev_k0, ev_k1, ev_total1;
    CUDA_CHECK(cudaEventCreate(&ev_total0));
    CUDA_CHECK(cudaEventCreate(&ev_k0));
    CUDA_CHECK(cudaEventCreate(&ev_k1));
    CUDA_CHECK(cudaEventCreate(&ev_total1));

    unsigned int grid = (unsigned int)((n + block - 1) / block);

    /* ---- región de interés con transferencias ---- */
    CUDA_CHECK(cudaEventRecord(ev_total0, 0));
    CUDA_CHECK(cudaMemcpy(d_in, h_in, bytes, cudaMemcpyHostToDevice));

    /* ---- región de interés sin transferencias (solo kernel) ---- */
    CUDA_CHECK(cudaEventRecord(ev_k0, 0));
    kernel_elemento<<<grid, (unsigned int)block>>>(d_in, d_out, (size_t)n, ITER_DEFAULT);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(ev_k1, 0));

    CUDA_CHECK(cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(ev_total1, 0));
    CUDA_CHECK(cudaEventSynchronize(ev_total1));

    float ms_kernel = 0.0f;
    float ms_total = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&ms_kernel, ev_k0, ev_k1));
    CUDA_CHECK(cudaEventElapsedTime(&ms_total, ev_total0, ev_total1));

    double cs = checksum(h_out, (size_t)n);

    printf("version=v3\n");
    printf("n=%lu\n", n);
    printf("semilla=%lu\n", seed);
    printf("gpu=%s\n", prop.name);
    printf("hilos_por_bloque=%lu\n", block);
    printf("tiempo_kernel_s=%.6f\n", (double)ms_kernel / 1000.0);
    printf("tiempo_total_s=%.6f\n", (double)ms_total / 1000.0);
    printf("checksum=%.12e\n", cs);

    CUDA_CHECK(cudaEventDestroy(ev_total0));
    CUDA_CHECK(cudaEventDestroy(ev_k0));
    CUDA_CHECK(cudaEventDestroy(ev_k1));
    CUDA_CHECK(cudaEventDestroy(ev_total1));
    CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_out));
    free(h_in);
    free(h_out);
    return 0;
}


In [ ]:
# 3. Compilación (las mismas banderas que en el repositorio)
!gcc -std=c11 -O2 -Wall -Wextra -I. -o v1 main_v1.c -lm
!nvcc -O2 -arch=sm_75 -Xcompiler -Wall,-Wextra -I. -o v3 kernel.cu
!ls -l v1 v3

In [ ]:
# 4. Verificación: V3 debe dar el mismo checksum que V1 con los mismos datos
import subprocess

TOL = 1e-9
N_PRUEBA, SEMILLA = 200000, 12345

def correr(cmd):
    out = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout
    return dict(l.split('=', 1) for l in out.strip().splitlines() if '=' in l)

r1 = correr(f'./v1 {N_PRUEBA} {SEMILLA}')
r3 = correr(f'./v3 {N_PRUEBA} {SEMILLA}')
c1, c3 = float(r1['checksum']), float(r3['checksum'])
ok = abs(c1 - c3) <= TOL * max(1.0, abs(c1))
print('v1 checksum =', r1['checksum'])
print('v3 checksum =', r3['checksum'], '| gpu =', r3.get('gpu', '?'))
print('PASS v3 == v1' if ok else 'FAIL v3 != v1: NO midas hasta corregirlo')
assert ok

In [ ]:
# 5. Medición (protocolo: 1 calentamiento + K repeticiones, mediana; con y sin transferencias)
import csv, statistics

N, SEMILLA, REPS = 4000000, 12345, 5

def medir(cmd, campo):
    correr(cmd)  # calentamiento, no se cuenta
    return [float(correr(cmd)[campo]) for _ in range(REPS)]

t_v1 = medir(f'./v1 {N} {SEMILLA}', 'tiempo_s')
t_k  = medir(f'./v3 {N} {SEMILLA}', 'tiempo_kernel_s')
t_t  = medir(f'./v3 {N} {SEMILLA}', 'tiempo_total_s')
T1 = statistics.median(t_v1)
if T1 < 1.0:
    print(f'AVISO: T(1) = {T1:.3f} s < 1 s en esta CPU; sube N (protocolo §2)')

cab = ['config', 'hilos', 'n'] + [f'rep{i}' for i in range(1, REPS + 1)] + ['mediana_s', 'speedup', 'eficiencia']
filas = []
for nombre, hilos, ts in (('v1', 1, t_v1), ('v3_kernel', 0, t_k), ('v3_total', 0, t_t)):
    med = statistics.median(ts)
    filas.append([nombre, hilos, N] + [f'{t:.6f}' for t in ts] + [f'{med:.6f}', f'{T1/med:.3f}', ''])
with open('mediciones_v3.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(cab); w.writerows(filas)
for fila in filas:
    print(f'{fila[0]:10s} mediana={fila[-3]} s  speedup_vs_v1={fila[-2]}')
print('CSV -> mediciones_v3.csv (hilos=0 marca filas de GPU; el speedup es contra la V1 de ESTA máquina)')

## Antes de cerrar la sesión

1. Revisa que las celdas 1, 4 y 5 tengan salida visible (nvidia-smi, PASS, medianas).
2. *Archivo → Descargar → .ipynb*: ese archivo **con salidas** es `v3/notebook_colab.ipynb` en el repositorio.
3. Descarga `mediciones_v3.csv` y `entorno_gpu.txt` a `resultados/` del repositorio y haz *commit*.
4. En el reporte, cita el modelo de GPU y la fecha que aparecen en `entorno_gpu.txt`.
